In [2]:
# ============================================================
# FILE 11 — HPA Description Exploration
# CellLineFinder | Person 2 | AstraZeneca MSc Dissertation
# ============================================================

import pandas as pd
import numpy as np

PATH = "./data/nomenclature/11_hpa_rna_celline_description.tsv"

# ============================================================
# BLOCK 1 — Detect separator, shape, columns
# ============================================================

# Try comma first, fall back to tab
try:
    df = pd.read_csv(PATH, nrows=3)
    sep = "," if df.shape[1] > 2 else None
    if sep is None:
        raise ValueError
except Exception:
    df = pd.read_csv(PATH, sep="\t", nrows=3)
    sep = "\t"

print(f"Separator: {repr(sep)}")

df = pd.read_csv(PATH, sep=sep, low_memory=False)
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumns:\n{list(df.columns)}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nFirst 3 rows:\n{df.head(3).to_string()}")

Separator: '\t'
Shape: 1,206 rows x 7 columns

Columns:
['Cell line', 'Disease', 'Disease subtype', 'Cellosaurus ID', 'Patient', 'Primary/Metastasis', 'Sample collection site']

Dtypes:
Cell line                 object
Disease                   object
Disease subtype           object
Cellosaurus ID            object
Patient                   object
Primary/Metastasis        object
Sample collection site    object
dtype: object

First 3 rows:
  Cell line          Disease Disease subtype Cellosaurus ID   Patient Primary/Metastasis Sample collection site
0      143B      Bone cancer    Osteosarcoma      CVCL_2270        13            Primary                   bone
1     22Rv1  Prostate cancer  Adenocarcinoma      CVCL_1045      Male            primary               prostate
2  23132/87   Gastric cancer  Adenocarcinoma      CVCL_1046  Male, 72            primary                stomach


In [3]:

# ============================================================
# BLOCK 2 — Nulls per column
# ============================================================

null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)
null_df = pd.DataFrame({"null_count": null_counts, "null_%": null_pct})
print(null_df.sort_values("null_%", ascending=False).to_string())


                        null_count  null_%
Primary/Metastasis             339    28.1
Disease subtype                159    13.2
Patient                        105     8.7
Sample collection site          68     5.6
Cellosaurus ID                   8     0.7
Cell line                        0     0.0
Disease                          0     0.0


In [4]:
# ============================================================
# BLOCK 3 — Cell line column: identify, count, sample values
# ============================================================

cl_candidates = [c for c in df.columns if any(
    kw in c.lower() for kw in ["cell", "line", "name", "sample", "id"]
)]
print(f"Candidate cell line columns: {cl_candidates}")

# Inspect each candidate
for col in cl_candidates:
    print(f"\n[{col}]")
    print(f"  Unique: {df[col].nunique():,}")
    print(f"  Sample: {df[col].dropna().head(15).tolist()}")
    has_hyphen = df[col].dropna().str.contains("-").sum()
    print(f"  Values with hyphens: {has_hyphen:,}")

Candidate cell line columns: ['Cell line', 'Cellosaurus ID', 'Sample collection site']

[Cell line]
  Unique: 1,206
  Sample: ['143B', '22Rv1', '23132/87', '253J', '253J-BV', '42-MG-BA', '537-mel', '5637', '59M', '624-mel', '639V', '647V', '697', '769-P', '786-O']
  Values with hyphens: 786

[Cellosaurus ID]
  Unique: 1,198
  Sample: ['CVCL_2270', 'CVCL_1045', 'CVCL_1046', 'CVCL_7935', 'CVCL_7937', 'CVCL_1798', 'CVCL_8052', 'CVCL_0126', 'CVCL_2291', 'CVCL_8054', 'CVCL_1048', 'CVCL_1049', 'CVCL_0079', 'CVCL_1050', 'CVCL_1051']
  Values with hyphens: 0

[Sample collection site]
  Unique: 46
  Sample: ['bone', 'prostate', 'stomach', 'lymph node', 'lymph node', 'central nervous system', 'urinary tract', 'ascites', 'urinary tract', 'urinary tract', 'bone marrow', 'kidney', 'kidney', 'central nervous system', 'thyroid']
  Values with hyphens: 0


In [5]:
# ============================================================
# BLOCK 4 — Sanity check: known cell lines present?
# ============================================================

# Adjust cl_col to whichever column you identified in Block 3
cl_col = cl_candidates[0]

known = {
    "A-431": ["A-431", "A431", "A431_SKIN"],
    "SK-BR-3": ["SK-BR-3", "SKBR3", "SKBR3_BREAST"],
    "HeLa": ["HeLa", "HELA", "HELA_CERVIX"],
    "MCF7": ["MCF7", "MCF-7"],
    "HEK293": ["HEK293", "HEK-293"],
    "SW480": ["SW480", "SW-480"],
    "H1299": ["H1299", "NCI-H1299"],
}

for label, variants in known.items():
    hits = df[df[cl_col].str.upper().isin([v.upper() for v in variants])]
    print(f"{label}: {len(hits)} row(s) found")
    if len(hits) > 0:
        print(hits[[cl_col]].head(3).to_string())

A-431: 1 row(s) found
   Cell line
25     A-431
SK-BR-3: 1 row(s) found
    Cell line
964   SK-BR-3
HeLa: 1 row(s) found
    Cell line
324      HeLa
MCF7: 1 row(s) found
    Cell line
584     MCF-7
HEK293: 1 row(s) found
    Cell line
321    HEK293
SW480: 1 row(s) found
     Cell line
1097     SW480
H1299: 1 row(s) found
     Cell line
676  NCI-H1299


In [6]:
# ============================================================
# BLOCK 5 — All low-cardinality columns: full value_counts
# ============================================================

for col in df.columns:
    n = df[col].nunique()
    if n <= 80:
        print(f"\n{'='*50}")
        print(f"[{col}]  ({n} unique values)")
        print(df[col].value_counts(dropna=False).head(30).to_string())


[Disease]  (30 unique values)
Disease
Lung cancer              232
Leukemia                  93
Brain cancer              80
Lymphoma                  76
Non-cancerous             63
Colorectal cancer         63
Breast cancer             62
Skin cancer               62
Ovarian cancer            59
Pancreatic cancer         46
Gastric cancer            42
Head and Neck cancer      38
Kidney cancer             35
Myeloma                   34
Uterine cancer            29
Esophageal cancer         27
Bladder cancer            26
Liver cancer              24
Bone cancer               21
Neuroblastoma             17
Sarcoma                   15
Rhabdoid                  14
Uncategorized             11
Thyroid cancer            11
Prostate cancer            8
Cervical cancer            8
Bile duct cancer           7
Adrenocortical cancer      1
Gallbladder cancer         1
Testis cancer              1

[Primary/Metastasis]  (4 unique values)
Primary/Metastasis
primary       457
NaN          

In [7]:
# ============================================================
# BLOCK 6 — High-cardinality columns: sample only
# ============================================================

for col in df.columns:
    n = df[col].nunique()
    if n > 80:
        print(f"\n[{col}]  ({n} unique) — sample:")
        print(df[col].dropna().head(10).tolist())


[Cell line]  (1206 unique) — sample:
['143B', '22Rv1', '23132/87', '253J', '253J-BV', '42-MG-BA', '537-mel', '5637', '59M', '624-mel']

[Disease subtype]  (121 unique) — sample:
['Osteosarcoma', 'Adenocarcinoma', 'Adenocarcinoma', 'Carcinoma', 'Carcinoma', 'Astrocytoma', 'Melanoma', 'Adenocarcinoma, high grade serous', 'Melanoma', 'Transitional Cell Carcinoma']

[Cellosaurus ID]  (1198 unique) — sample:
['CVCL_2270', 'CVCL_1045', 'CVCL_1046', 'CVCL_7935', 'CVCL_7937', 'CVCL_1798', 'CVCL_8052', 'CVCL_0126', 'CVCL_2291', 'CVCL_8054']

[Patient]  (200 unique) — sample:
['13', 'Male', 'Male, 72', 'Male, 53', 'Male, 53', 'Male, 63', 'Male, 68', 'Female, 65', 'Male, 69', 'Male, 59']


In [8]:
# ============================================================
# BLOCK 7 — Tissue / lineage / cancer type columns
# ============================================================

tissue_cols = [c for c in df.columns if any(
    kw in c.lower() for kw in [
        "tissue", "cancer", "organ", "lineage", "disease",
        "type", "origin", "tumor", "site", "category"
    ]
)]
print(f"Tissue/lineage candidates: {tissue_cols}")

for col in tissue_cols:
    print(f"\n[{col}] — {df[col].nunique()} unique values:")
    print(df[col].value_counts(dropna=False).head(25).to_string())

Tissue/lineage candidates: ['Disease', 'Disease subtype', 'Sample collection site']

[Disease] — 30 unique values:
Disease
Lung cancer             232
Leukemia                 93
Brain cancer             80
Lymphoma                 76
Non-cancerous            63
Colorectal cancer        63
Breast cancer            62
Skin cancer              62
Ovarian cancer           59
Pancreatic cancer        46
Gastric cancer           42
Head and Neck cancer     38
Kidney cancer            35
Myeloma                  34
Uterine cancer           29
Esophageal cancer        27
Bladder cancer           26
Liver cancer             24
Bone cancer              21
Neuroblastoma            17
Sarcoma                  15
Rhabdoid                 14
Uncategorized            11
Thyroid cancer           11
Prostate cancer           8

[Disease subtype] — 121 unique values:
Disease subtype
NaN                                                            159
Adenocarcinoma                                        

In [9]:
# ============================================================
# BLOCK 8 — CVCL / accession / DepMap ID columns
# ============================================================

id_cols = [c for c in df.columns if any(
    kw in c.lower() for kw in [
        "cvcl", "accession", "cellosaurus", "ach", "depmap",
        "ccle", "broad", "model"
    ]
)]
print(f"ID/accession candidates: {id_cols}")

for col in id_cols:
    print(f"\n[{col}] — {df[col].nunique()} unique values")
    print(f"  Sample: {df[col].dropna().head(10).tolist()}")
    cvcl_hits = df[col].dropna().str.contains("CVCL", case=False).sum()
    ach_hits = df[col].dropna().str.contains("ACH-", case=False).sum()
    print(f"  Contains CVCL: {cvcl_hits}")
    print(f"  Contains ACH-: {ach_hits}")

ID/accession candidates: ['Cellosaurus ID']

[Cellosaurus ID] — 1198 unique values
  Sample: ['CVCL_2270', 'CVCL_1045', 'CVCL_1046', 'CVCL_7935', 'CVCL_7937', 'CVCL_1798', 'CVCL_8052', 'CVCL_0126', 'CVCL_2291', 'CVCL_8054']
  Contains CVCL: 1198
  Contains ACH-: 0


In [10]:

# ============================================================
# BLOCK 9 — Name normalisation: strip hyphens/spaces/case
#           Check overlap with known File 1 cell line count (1,206)
# ============================================================

df["_norm"] = df[cl_col].str.replace(r"[\s\-]", "", regex=True).str.upper()

print(f"Total rows: {len(df):,}")
print(f"Unique raw names: {df[cl_col].nunique():,}")
print(f"Unique normalised names: {df['_norm'].nunique():,}")
print(f"\nSample normalised: {df['_norm'].head(15).tolist()}")

# How many normalised names match the File 1 sanity set?
file1_norm = {"A431", "SKBR3", "HELA", "MCF7", "HEK293", "SW480", "H1299"}
overlap = df["_norm"].isin(file1_norm).sum()
print(f"\nRows matching File 1 sanity set: {overlap}")

Total rows: 1,206
Unique raw names: 1,206
Unique normalised names: 1,206

Sample normalised: ['143B', '22RV1', '23132/87', '253J', '253JBV', '42MGBA', '537MEL', '5637', '59M', '624MEL', '639V', '647V', '697', '769P', '786O']

Rows matching File 1 sanity set: 6


In [11]:
# ============================================================
# BLOCK 10 — Duplicate detection on cell line column
# ============================================================

dupes = df[cl_col].value_counts()
dupes_multi = dupes[dupes > 1]
print(f"Cell line names appearing more than once: {len(dupes_multi)}")
if len(dupes_multi) > 0:
    print(dupes_multi.head(20).to_string())

# Check if duplicates differ in other columns (e.g. tissue)
if len(dupes_multi) > 0:
    example = dupes_multi.index[0]
    print(f"\nExample duplicate — '{example}':")
    print(df[df[cl_col] == example].to_string())

Cell line names appearing more than once: 0


In [12]:
# ============================================================
# BLOCK 11 — HPA-specific metadata fields
# ============================================================

hpa_cols = [c for c in df.columns if any(
    kw in c.lower() for kw in [
        "morph", "growth", "sex", "age", "ethn", "grade",
        "stage", "url", "source", "passage", "karyotype"
    ]
)]
print(f"HPA metadata candidates: {hpa_cols}")

for col in hpa_cols:
    n = df[col].nunique()
    print(f"\n[{col}] — {n} unique values:")
    print(df[col].value_counts(dropna=False).head(20).to_string())

HPA metadata candidates: []


In [13]:
# ============================================================
# BLOCK 12 — Recommended Neo4j CellLine node properties
#            Print a summary to fill in after reviewing above
# ============================================================

print("""
SUMMARY — columns to carry into Neo4j CellLine node:

Review Block 5/6/7/8/11 output and fill in:

  Golden key    : CVCL accession (from Block 8 if present, else join via File 7)
  Human name    : <whichever cell line name column you found>
  Tissue        : <tissue/organ column>
  Cancer type   : <disease/cancer type column>
  Morphology    : <morphology column if present>
  Sex           : <sex column if present>
  HPA URL       : <url column if present>

Any column >80% null → drop or store as optional property only.
""")


SUMMARY — columns to carry into Neo4j CellLine node:

Review Block 5/6/7/8/11 output and fill in:

  Golden key    : CVCL accession (from Block 8 if present, else join via File 7)
  Human name    : <whichever cell line name column you found>
  Tissue        : <tissue/organ column>
  Cancer type   : <disease/cancer type column>
  Morphology    : <morphology column if present>
  Sex           : <sex column if present>
  HPA URL       : <url column if present>

Any column >80% null → drop or store as optional property only.



In [14]:

# ============================================================
# BLOCK 13 — Export exploration summary to HTML
# ============================================================

summary = {
    "Shape": f"{df.shape[0]:,} rows x {df.shape[1]} cols",
    "Separator": sep,
    "Cell line column": cl_col,
    "Unique cell lines (raw)": df[cl_col].nunique(),
    "Unique cell lines (normalised)": df["_norm"].nunique(),
    "Columns": list(df.columns),
}

null_summary = (
    df.isnull().sum()
    .rename("null_count")
    .to_frame()
    .assign(null_pct=(df.isnull().sum() / len(df) * 100).round(1))
)

with open("file11_hpa_description_reference.html", "w") as f:
    f.write("<html><head><title>File 11 HPA Description Reference</title>")
    f.write("<style>body{font-family:monospace;padding:2rem} table{border-collapse:collapse} ")
    f.write("td,th{border:1px solid #ccc;padding:4px 8px;font-size:13px} ")
    f.write("h2{margin-top:2rem}</style></head><body>")
    f.write("<h1>File 11 — HPA Description Reference</h1>")

    f.write("<h2>Summary</h2><table>")
    for k, v in summary.items():
        f.write(f"<tr><td><b>{k}</b></td><td>{v}</td></tr>")
    f.write("</table>")

    f.write("<h2>Null Summary</h2>")
    f.write(null_summary.to_html())

    f.write("<h2>First 5 Rows</h2>")
    f.write(df.head(5).to_html())

    f.write("<h2>Value Counts — Low Cardinality Columns</h2>")
    for col in df.columns:
        if df[col].nunique() <= 80:
            f.write(f"<h3>{col}</h3>")
            f.write(df[col].value_counts(dropna=False).head(30).to_frame().to_html())

    f.write("</body></html>")

print("Exported: file11_hpa_description_reference.html")

Exported: file11_hpa_description_reference.html


In [15]:
missing_cvcl = df[df["Cellosaurus ID"].isnull()]
print(missing_cvcl[["Cell line", "Disease", "Sample collection site"]].to_string())

                           Cell line        Disease Sample collection site
38                              AF22  Uncategorized                    NaN
45           ASC2telo differentiated  Uncategorized                    NaN
68           BJ hTERT+ SV40 Large T+  Uncategorized                    NaN
69   BJ hTERT+ SV40 Large T+ RasG12V  Uncategorized                    NaN
331                           HHSteC  Uncategorized                    NaN
394                            HSkMC  Uncategorized                    NaN
892                   PODO/SVTERT152  Non-cancerous                    NaN
893                     PODO/TERT256  Non-cancerous                    NaN


In [16]:
# Load File 1 cell line names and check direct overlap
file1_cls = set(pd.read_csv("./data/gene expression/1_4_hpa_rna_celline.tsv", sep="\t", usecols=["Cell line"])["Cell line"].unique())
file11_cls = set(df["Cell line"].unique())
print(f"Exact match overlap: {len(file1_cls & file11_cls)}")
print(f"Only in File 1: {len(file1_cls - file11_cls)}")
print(f"Only in File 11: {len(file11_cls - file1_cls)}")

Exact match overlap: 1206
Only in File 1: 0
Only in File 11: 0
